# 04. Player Absence & Injury Classification

**Stage:** 04_adjustments  
**Inputs:** Player match logs, `src/config/config.yaml`  
**Outputs:** `data/processed/absences.parquet`  

This notebook identifies key starters per team based on rolling minute share thresholds (45%+ minute share) and classifies player availability/absences into injury vs rotation/suspension tags without forward data leakage.

In [1]:
# Load imports and configuration
%load_ext autoreload
%autoreload 2
from pathlib import Path
import sys

import pandas as pd

repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "src" / "config" / "loader.py").exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root))

from src.config.loader import load_config
from src.adjustments.absence_classifier import identify_player_absences

config = load_config()
PROJECT_ROOT = Path.cwd().resolve().parents[1]
processed_dir = PROJECT_ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

player_logs_df = None
absences_df = None
print("Absence classification stage initialized.")

Absence classification stage initialized.


In [2]:
# Load or initialize player match logs
# Note: Full player availability logs are constructed from player match pages
sample_file = processed_dir / "player_match_logs.parquet"
if sample_file.exists():
    player_logs_df = pd.read_parquet(sample_file)
    print(f"Loaded player match logs: {len(player_logs_df)} rows")
else:
    player_logs_df = pd.DataFrame(columns=["fixture_id", "date", "team", "player_id", "player_name", "position", "minutes_played"])
    print("No pre-saved player match logs found; initialized empty structure for absence pipeline")

No pre-saved player match logs found; initialized empty structure for absence pipeline


In [3]:
# Classify player absences for key starters (>=45% rolling minute share)
if player_logs_df is not None and not player_logs_df.empty:
    absences_df = identify_player_absences(player_logs_df, key_starter_threshold=0.45, rolling_window=5)
    print(f"Identified {len(absences_df)} key starter absences")
else:
    absences_df = pd.DataFrame(columns=["fixture_id", "date", "team", "player_id", "player_name", "position_group", "absence_type", "starter_minute_share"])
    print("Initialized empty absences table")

Initialized empty absences table


In [4]:
# Absence Summary & Analysis
if absences_df is not None and not absences_df.empty:
    print("=== Absence Type Distribution ===")
    print(absences_df["absence_type"].value_counts())
    print("\n=== Absence by Position Group ===")
    print(absences_df["position_group"].value_counts())
    display(absences_df.head())
else:
    print("Absence summary: 0 recorded starter absences in current scope")

Absence summary: 0 recorded starter absences in current scope


In [5]:
# Save absences dataset
if absences_df is not None:
    output_file = processed_dir / "absences.parquet"
    absences_df.to_parquet(output_file, index=False)
    print(f"Saved absences to {output_file}")

Saved absences to /Users/mac/Documents/Projects/SportsBettingPoisson+ML/data/processed/absences.parquet
